# KCS 2025 October 1-20 PDF source comparability
One exact official-portal read, no SOXX outcomes.


In [1]:
import datetime,hashlib,json,shlex,subprocess,sys,tarfile,tempfile
from pathlib import Path
root=Path.cwd() if (Path.cwd()/'src').exists() else Path.cwd().parents[1];sys.path.insert(0,str(root))
from src.data.nas_market_data import nas_api_call,_settings,_ssh
card=root/'research/kcs_early20_2025_oct_comparability_20261023';raw=root/'data/snapshots/kcs_early20_2025_oct_comparability_20261023';raw.mkdir(parents=True,exist_ok=True)
plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest();assert plan['frozen'] and not (card/'run_state.json').exists()
assert sha(root/plan['current_2026_reference'])==plan['current_2026_reference_sha256']
(card/'run_state.json').write_text(json.dumps({'status':'started','plan_sha256':sha(card/'plan.json'),'GETs_started':1},indent=2)+'\n')
body=raw/'attachment.body';headers=raw/'attachment.headers';start=datetime.datetime.now(datetime.timezone.utc).isoformat()
proc=subprocess.run(['curl','--location','--silent','--show-error','--connect-timeout','5','--max-time','20','--dump-header',str(headers),'--output',str(body),'--write-out','%{http_code}|%{url_effective}',plan['source_url']],capture_output=True,text=True)
end=datetime.datetime.now(datetime.timezone.utc).isoformat();pieces=proc.stdout.strip().split('|',1)
receipt={'source_url':plan['source_url'],'started_at_utc':start,'completed_at_utc':end,'curl_exit':proc.returncode,'http_status':pieces[0] if pieces else None,'effective_url':pieces[1] if len(pieces)>1 else None,'error':proc.stderr[:300] if proc.returncode else None,'body_path':str(body.relative_to(root)),'body_bytes':body.stat().st_size if body.exists() else 0,'body_sha256':sha(body) if body.exists() else None,'headers_path':str(headers.relative_to(root)),'headers_bytes':headers.stat().st_size if headers.exists() else 0,'headers_sha256':sha(headers) if headers.exists() else None}
(card/'response_receipt.json').write_text(json.dumps(receipt,ensure_ascii=False,indent=2)+'\n')
valid=receipt['http_status']=='200' and body.exists() and body.read_bytes().startswith(b'%PDF-');info=None
if valid:
 checked=subprocess.run(['pdfinfo',str(body)],capture_output=True,text=True,timeout=20)
 if checked.returncode==0:info={'pages':next((int(x.split(':',1)[1].strip()) for x in checked.stdout.splitlines() if x.startswith('Pages:')),None),'pdfinfo_exit':0}
 else:valid=False;info={'pdfinfo_exit':checked.returncode,'error':checked.stderr[:300]}
result={'status':'PDF_bytes_valid_visual_comparison_pending' if valid else 'needs_source_transport_or_PDF_validation','receipt':receipt,'valid_PDF':valid,'pdfinfo':info,'2025_semiconductor_amount_USD_million':None,'year_over_year_reconciliation':None,'historical_first_public_bytes_qualified':False,'strategy_forward_day_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
entries={str(p.relative_to(root)):sha(p) for p in sorted(raw.iterdir()) if p.is_file()};entries[str((card/'plan.json').relative_to(root))]=sha(card/'plan.json');entries[str((card/'response_receipt.json').relative_to(root))]=sha(card/'response_receipt.json')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='kcs-2025-pdf-') as temp:
 archive=Path(temp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as out:
  for rel in sorted(entries):out.add(root/rel,arcname=rel,recursive=False)
 digest=sha(archive);size=archive.stat().st_size;evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact'];assert evidence['sha256']==digest and evidence['size_bytes']==size
 nas_path=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest;remote=_ssh(settings['ssh'],'sha256sum '+shlex.quote(nas_path)).split()[0];assert remote==digest
 with tempfile.TemporaryDirectory(prefix='kcs-2025-readback-') as back:
  fetched=Path(back)/'restore.tar.gz'
  with fetched.open('wb') as output:
   call=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes','-o','ConnectTimeout=5',settings['ssh']['host'],'cat '+shlex.quote(nas_path)],stdout=output,stderr=subprocess.PIPE,timeout=90)
  assert call.returncode==0 and sha(fetched)==digest
  with tarfile.open(fetched,'r:gz') as source:
   members={m.name:m for m in source.getmembers()};assert set(entries)==set(members)
   for rel,expected in entries.items():
    f=source.extractfile(members[rel]);assert f and hashlib.sha256(f.read()).hexdigest()==expected
archive_receipt={'status':'uploaded_exact_2025_official_portal_PDF_response','archive_sha256':digest,'archive_bytes':size,'entries':entries,'nas_artifact':evidence,'nas_path':nas_path,'remote_hash_verified':True,'scoped_restore_verified':True}
(card/'source_archive_receipt.json').write_text(json.dumps(archive_receipt,ensure_ascii=False,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':result['status'],'objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'result':'result.json','source_archive_receipt':'source_archive_receipt.json','source_NAS_sha256':digest,'source_NAS_scoped_restore_verified':True,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':[]}}
(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n');(card/'run_state.json').write_text(json.dumps({'status':'completed_transport_visual_pending' if valid else 'completed_transport_failed','plan_sha256':sha(card/'plan.json'),'GETs_completed':1,'NAS_scoped_restore_verified':True},indent=2)+'\n')
print(json.dumps({'result_status':result['status'],'receipt':receipt,'NAS_sha256':digest},ensure_ascii=False))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{"result_status": "PDF_bytes_valid_visual_comparison_pending", "receipt": {"source_url": "https://www.korea.kr/common/download.do?fileId=198220303&tblKey=GMN", "started_at_utc": "2026-10-23T00:44:38.578028+00:00", "completed_at_utc": "2026-10-23T00:44:46.680656+00:00", "curl_exit": 0, "http_status": "200", "effective_url": "https://www.korea.kr/common/download.do?fileId=198220303&tblKey=GMN", "error": null, "body_path": "data/snapshots/kcs_early20_2025_oct_comparability_20261023/attachment.body", "body_bytes": 1395010, "body_sha256": "093060193d2557137ecc933584756c410506288feeb395c496586b68e659d8ef", "headers_path": "data/snapshots/kcs_early20_2025_oct_comparability_20261023/attachment.headers", "headers_bytes": 509, "headers_sha256": "cce93a930b331cbdd9f699c867a561ca757ef2ca50d1ecb887295fa50c3e387b"}, "NAS_sha256": "a7fb2a8d69837fabebaae357cadc3cf7e760d0b14dd6bdf897afdf8b7660a850"}


# 韩国海关2025年10月前20日：当前PDF跨年口径核对

一次事前冻结的官方门户 PDF 抓取获得 **HTTP 200、1,395,010 字节**，SHA-256 `093060193d2557137ecc933584756c410506288feeb395c496586b68e659d8ef`；四页均渲染审阅。第4页的 2025 年 10 月 1–20 日出口表半导体行是 **8,533 百万美元，同比 +20.2%**，与 2026 年同期间 PDF 的 27,516 百万美元、+222.5% 使用相同类别和单位。按预先固定的除法，`27,516 / 8,533 - 1` 为 **222.4657%**，取一位小数即 +222.5%。

关键时序限制：2025 PDF 第1页印有 **2025-10-21 09:00 韩国时间即刻分发**，当前 PDF 的创建元数据显示 **2025-10-22 14:18:19 CST**，晚于印发时刻。故两份**现存** PDF 的数值对齐，只能证明这对样本没有显见的单位/同比基数冲突，不能证明 2025 PDF 当前字节或金额是当日首次公开版本。2025 同期间工作日 10.5、2026 为 13.0，原始同比还受工作日差异影响；不得直接称为需求意外。

原始 PDF、响应头及精确哈希经 NAS 归档 SHA-256 `a7fb2a8d69837fabebaae357cadc3cf7e760d0b14dd6bdf897afdf8b7660a850`，远端哈希和限定恢复通过。本卡 **0 SOXX 结果读取、0 拟合、0 账户、0 策略前向日**。继续历史来源需找到当时版本；11月正式月报的来源抓取已另行冻结，之后也须先处理单位、工作日及修订再定义任何 Stage1 特征。
